### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/14_CIFAR100_Joint_HPO.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F14_CIFAR100_Joint_HPO.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F14_CIFAR100_Joint_HPO.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/14_CIFAR100_Joint_HPO.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.


See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.

Kaggle import fallback: open Kaggle Code → Import Notebook → Local file, upload this notebook, then choose Quick Save.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, revision=REVISION, 
    runtime=RUNTIME, cuda=CUDA
)

# CIFAR100: joint diffusion/classifier HPO

All classes train together, with a `new_weight` class token, **EMA**, and no distillation.
The shared API maximizes **accuracy** and minimizes **noise loss** as separate Optuna objectives.

**Data option:** these settings fit on **all 50,000 official training examples** and use
**all 10,000 official test examples for fit validation and HPO**. No training rows are reserved.
Choose `VALIDATION_SOURCE = "split"` and a positive `VALIDATION_RATIO` to use an internal split instead. With the default test-set reuse, scores are
tuning results, not an independent final test. This search is separate from the frozen thesis campaign.

See [the HPO guide](JOINT_CLASSIFIER_HPO.md) for the complete recipe, report findings, and limitations.

In [ ]:
from common.hpo import run_hpo, summarize_hpo
from common.hpo_profiles import JOINT_CLASSIFIER_SEARCH_SPACE
from notebooks.thesis.workflow import check_runtime


print(check_runtime())
JOINT_CLASSIFIER_SEARCH_SPACE

## 1. Budget

Batch 128 and chunk size 16 are starting points for A100/H100, not measured memory guarantees.
V1 allows 50 joint epochs; V2 allows **50 generator + 50 classifier epochs**.
Each phase uses early stopping (patience 10) and plateau LR reduction (patience 5).

Run All continues the remaining **total allocated trial budget**, including failed/pruned attempts.
Preserve the results directory between hosted sessions. Change `STUDY_ROOT` if scientific settings change;
Version 5 uses a new directory after restoring the native denoising timestep defaults. Run only one kernel per study.

In [ ]:
DATASET = "cifar100"
VALIDATION_SOURCE = "test"  # "test": full train/test; "split": internal validation.
VALIDATION_RATIO = 0.0  # Used only for "split"; set e.g. 0.2 in that mode.
SEED = 17
TOTAL_TRIALS = 120
STARTUP_TRIALS = 30
EPOCHS = 50
BATCH_SIZE = 128
DTYPE_POLICY = "mixed_bfloat16"
ENSEMBLE_CHUNK_SIZE = 16
TIMEOUT_HOURS = 24  # Checked between trials; None removes the limit.
STUDY_ROOT = ROOT / "results/thesis_route_one/joint_classifier_hpo_v5"
STUDY_DIRECTORY = STUDY_ROOT / "joint/dit_classifier" / DATASET / "joint_dit_classifier"
TENSORBOARD_DIRECTORY = STUDY_DIRECTORY / "tensorboard"

## 2. Run or continue

- V1 searches `clf_loss_coef=[0.001, 0.01, 0.1, 0.25, 0.5, 1.]`; V2 fixes it to 1.
- `modify_first_t` is searched; EMA and the null mask are enabled. Native classifier-training and CFG-scale defaults remain.
- Accuracy uses **EMA ensemble accuracy** for V1 and positive-cap V2; clean V2 uses ordinary EMA accuracy.
  V2 ensemble `max_t` equals its positive classifier cap. V1 uses 128 timesteps.
- Early stopping selects ordinary EMA accuracy (EMA noise loss in V2's generator phase).
  Final objectives come from the same restored model; no weighted sum hides either objective.
- Final denoising scores use native default timesteps 0-999 with fixed random draws per split.
  `modify_first_t` retains its native timestep-zero behavior.
  Classifier ensembles retain their specified timesteps; final evaluation preserves training RNG state.
- NaN/Inf losses or final objectives prune a trial. OOM is recorded as failed. Finite underperformers use early stopping;
  standard Optuna median/Hyperband pruning does not support this multi-objective study.
- Every completed trial saves weights, CSVs, plots, TensorBoard logs, and four EMA image grids/GIFs
  including the null class. The two 1000-step modes can make final reporting expensive, especially on CIFAR-100.

In [ ]:
study = run_hpo(
    task="joint",
    model_name="dit_classifier",
    dataset_name=DATASET,
    validation_source=VALIDATION_SOURCE,
    validation_ratio=VALIDATION_RATIO,
    search_profile="joint_dit_classifier",
    n_trials=TOTAL_TRIALS,
    trial_budget_mode="total",
    n_startup_trials=STARTUP_TRIALS,
    epochs=EPOCHS,
    seed=SEED,
    results_path=str(STUDY_ROOT),
    timeout=None if TIMEOUT_HOURS is None else TIMEOUT_HOURS * 3600,
    objective_metrics=["classification_accuracy", "noise_loss"],
    objective_directions=["maximize", "minimize"],
    dtype_policy=DTYPE_POLICY,
    search_space_overrides={"batch_size": [BATCH_SIZE]},
    ensemble_accuracy_kwargs={"t_chunk_size": ENSEMBLE_CHUNK_SIZE},
)

## 3. Pareto results

Each row is a completed trial that no other trial improves in both objectives.
Accuracy is a fraction; lower noise loss is better. There is no single automatic winner.
Inspect generated images too: noise MSE is not a perceptual-quality metric.
`summarize_hpo(study, pareto_only=False)` includes failed/pruned attempts.

In [ ]:
summarize_hpo(study)

## 4. TensorBoard

Epoch metrics and both final objectives are saved per trial, with separate V2 phase logs.
The study directory also contains `trials.csv`, `pareto_trials.csv`, SQLite state and trial configurations.
Each successful run has `model.weights.h5`, evaluation/history CSVs, history plots and all four sampling modes.
Aborted trials retain their available logs/configuration and divergence evidence, not fabricated final artifacts.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_DIRECTORY --port 6006